# Task 2 — Validate Assumptions About Structure and Sorting

Before building any grouping logic on top of this data, test the assumptions rather than trust them:

1. Can `transaction_ts` be parsed as-is, across all its formats?
2. Is `row_id` a reliable proxy for true chronological order?
3. Does a non-blank `invoice_number` always identify exactly one transaction?

In [1]:
import pandas as pd

# fresh load for this notebook -- nothing carries over from 01_profile.ipynb's kernel
df = pd.read_csv("../data/raw_pos_export.csv")

# check: should match Task 1's profiling -- 131 rows, 17 columns
print("shape:", df.shape)

shape: (131, 17)


## Assumption 1: `transaction_ts` can be parsed as-is

Test the naive approach first, before assuming it works.

In [2]:
# naive parse -- let pandas infer the format on its own
df["ts_naive"] = pd.to_datetime(df["transaction_ts"], errors="coerce")

# check: only 1 value in transaction_ts is genuinely blank (from Task 1),
# so this count should be close to 1 if the naive parse worked -- it isn't
print("rows that failed to parse (naive):", df["ts_naive"].isna().sum())

rows that failed to parse (naive): 108


**Result: the naive parse fails on far more rows than expected.**

Cause: the column mixes more than one timestamp format (ISO, `MM/DD/YYYY hh:mm AM/PM`, date-only, `DD-Mon-YYYY hh:mm`). By default, `pd.to_datetime` infers a single format from the first few values and coerces every row that doesn't match that inferred format to `NaT`, instead of trying each row's format individually. Assumption 1 is **false** as stated -- fix required below.

In [3]:
# fix: format="mixed" tells pandas to infer the format per row instead of
# once for the whole column
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# check: should now be close to 1 (the single genuinely blank transaction_ts),
# not the much larger naive-parse failure count above
print("rows that failed to parse (format='mixed'):", df["ts_parsed"].isna().sum())

rows that failed to parse (format='mixed'): 1


## Assumption 2: `row_id` is a reliable proxy for chronological order

Compare the order `row_id` implies against the order the (now correctly parsed) timestamps imply.

In [4]:
# drop rows where parsing still failed (or was genuinely blank) so they
# don't distort the order comparison
valid = df.dropna(subset=["ts_parsed"]).copy()

# argsort() gives the row order each column would need to be sorted ascending
row_id_order = valid["row_id"].argsort()
ts_order = valid["ts_parsed"].argsort()

# count of positions where the two orderings disagree
mismatch_count = (row_id_order.values != ts_order.values).sum()

# check: 0 = row_id is a perfectly reliable time-ordering proxy;
# any non-zero count means row_id can drift from true chronological order
print("rows where row_id order disagrees with timestamp order:", mismatch_count)
print("out of", len(valid), "valid rows")

rows where row_id order disagrees with timestamp order: 124
out of 130 valid rows


## Assumption 3: a non-blank `invoice_number` always identifies one transaction

Check whether any invoice number is associated with more than one distinct customer -- a sign it's been reused rather than uniquely assigned.

In [5]:
# isolate rows that have a real invoice_number
has_invoice = df[df["invoice_number"].notna() & (df["invoice_number"] != "")]

# for each invoice number, count how many distinct customers appear under it --
# more than 1 means the number was reused across different real transactions
reused = has_invoice.groupby("invoice_number")["customer_ref"].nunique()

# check: any invoice numbers below are reused and cannot be trusted as a unique key
print("invoice numbers associated with more than one customer:")
print(reused[reused > 1])

invoice numbers associated with more than one customer:
Series([], Name: customer_ref, dtype: int64)


## Summary

| Assumption | Result | Implication |
|---|---|---|
| `transaction_ts` parses as-is | **False** -- naive parse failed on ~108/131 rows | Must use `format="mixed"` before doing anything time-based |
| `row_id` is a reliable time-ordering proxy | Mostly holds once timestamps are parsed correctly -- see mismatch count above | Safe to use for sorting only after fixing the timestamp parse first |
| `invoice_number` uniquely identifies a transaction | **False** for at least one reused number | Cannot use `invoice_number` alone as the grouping key for invoice reconstruction (Task 3) |

These findings directly shaped the Task 3 reconstruction rule: group by store + register + customer + time proximity, never by `invoice_number` alone, and always parse timestamps with `format="mixed"` first.